[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/097.%20The%20Last-Mile%20Delivery%20%26%20Micro-Fulfillment%20Problem/P97-Tier-10_executed.ipynb)

---


# 97. The Last-Mile Delivery & Micro-Fulfillment Problem
## Tier 10: Proactive Demand and Market Shaping

### Purpose and scope

This notebook builds a small, synthetic delivery-market scenario, estimates its hourly demand, and applies specified demand-shaping changes to compare demand and simple operating metrics before and after the intervention. The implementation is a simulation: it is not a demand-forecasting model, a customer-choice model, or an optimization procedure. The intervention percentages are scenario assumptions, not effects estimated from customer behavior.

The inputs are generated customer profiles, their hourly base-demand patterns, and three illustrative incentive mechanisms. The outputs are hourly demand estimates and comparisons of demand, vehicle requirements, estimated vehicle costs, and an emissions proxy. Random noise is used in the demand analysis, so results can differ between notebook runs; no random seed is set.

### Model assumptions

- The city is represented by a bounded coordinate space. Customer locations and vehicle locations are illustrative; the calculations below do not use travel distances or route plans.
- Each customer has a 24-hour base-demand profile and behavioral scores on a 0-to-1 scale. Hourly demand is perturbed by random noise, adjusted by hour-specific and customer-specific factors, and bounded below and above.
- The holiday-demand multiplier is 3.4. This means demand is scaled to 3.4 times its unmultiplied level, which is a 240% increase over that level. A 340% increase would instead mean 4.4 times the level; the separately cited 340% claim is not modeled as such here.
- The intervention applies a 43% reduction to demand in the 4:00–6:00 PM hourly bins and a 28% increase to demand in all other hourly bins. These are independent scaling assumptions: the model does not require reduced peak demand to reappear in off-peak hours.
- Vehicle requirements are approximated using a fixed capacity of 80 packages per vehicle. Estimated vehicle cost is $500 per required vehicle. Emissions are represented by a stated proxy rather than measured fuel use or route emissions.
- Customer satisfaction, adoption rates, actual savings, and real-world emissions are not calculated from customer behavior or operational data. They cannot be inferred from this simulation.

### Approach

1. Define data structures for customers, delivery options, incentives, and vehicles.
2. Generate a synthetic customer base and estimate baseline hourly demand.
3. Apply the specified peak-period reduction and off-peak increase.
4. Compare demand totals and simple vehicle, cost, and emissions-proxy metrics.
5. Visualize and interpret the simulated results, keeping scenario assumptions distinct from measured outcomes.

### Mathematical representation

Let \(i\) index customers and \(h\in\{0,1,\ldots,23\}\) index hourly bins. Let \(b_{i,h}\) be customer \(i\)'s base demand, and let \(\epsilon_{i,h}\) be a random perturbation drawn uniformly from \([-0.2,0.2]\). The modeled customer demand before aggregation is

$$
q_{i,h}=\min\!\left(5,\max\!\left(0.1,
\left(b_{i,h}+\epsilon_{i,h}\right)
\,p_h\,
(1+0.2s_i)(1+0.1t_i)(1+0.1l_i)(1+0.1r_i)
\right)\right),
$$

where \(s_i\), \(t_i\), \(l_i\), and \(r_i\) are the customer's price-sensitivity, time-preference, loyalty, and responsiveness scores. The hourly factor \(p_h\) is 1.5 for hours 8, 9, 17, and 18; 0.8 for hours 13 and 14; and 1 otherwise. Aggregate baseline demand is \(D_h=\sum_i q_{i,h}\). The code implements this calculation in `MarketShaper.analyze_demand_patterns`.

Let \(P=\{16,17\}\) denote the two hourly bins from 4:00 PM to 6:00 PM, and let \(O=\{0,\ldots,23\}\setminus P\). The intervention produces

$$
D'_h=\begin{cases}
(1-0.43)D_h, & h\in P,\\
(1+0.28)D_h, & h\in O.
\end{cases}
$$

The targeted-period reduction and the change in demand outside that period are calculated from period totals. If \(D_P=\sum_{h\in P}D_h\) and \(D_O=\sum_{h\in O}D_h\), then

$$
R_P=100\frac{D_P-D'_P}{D_P},
\qquad
G_O=100\frac{D'_O-D_O}{D_O}.
$$

These measures describe the assumed intervention's effect on the corresponding groups of hourly bins. They are not the change in the maximum or minimum hourly demand across the whole day, which can differ when the identity of the busiest or quietest hour changes.

For total demand \(D_{\mathrm{total}}\), the vehicle estimate is \(V=\lceil D_{\mathrm{total}}/80\rceil\). Estimated savings are \((V_{\mathrm{before}}-V_{\mathrm{after}})\times 500\) dollars. The emissions proxy is \(0.5R_P\) percentage points. These are deliberately simple calculations, not a routing model or an emissions inventory.

### Interpreting the results

The comparison shows what follows from the stated demand multipliers and operating assumptions. It does not establish that customers adopt the incentives, that demand is forecast accurately, or that the cited business outcomes will occur. The figures in the source scenario—such as a 43% peak reduction, $18,200 in savings, a 22% emissions reduction, and 94.6% customer satisfaction—are reference claims, not results validated by this synthetic simulation.

In [ ]:
from dataclasses import dataclass
from datetime import datetime
import random
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Customer:
    """A customer profile used by the synthetic demand simulation."""

    id: int
    location: Tuple[float, float]
    base_demand_pattern: List[float]
    price_sensitivity: float
    time_preference: float
    loyalty_score: float
    responsiveness: float
    demographic_profile: Dict[str, Any]


@dataclass
class DeliveryOption:
    """A delivery option that could be presented to a customer."""

    id: int
    delivery_time: float
    price: float
    convenience_score: float
    environmental_impact: float
    capacity_utilization: float
    social_proof: List[str]


@dataclass
class IncentiveMechanism:
    """An illustrative incentive and its scenario parameters."""

    id: int
    name: str
    type: str
    target_segment: List[str]
    incentive_value: float
    time_window: Tuple[float, float]
    expected_impact: float
    implementation_cost: float


@dataclass
class Vehicle:
    """A delivery vehicle with illustrative capacity and cost data."""

    id: int
    capacity: float
    cost_per_km: float
    location: Tuple[float, float]


class MarketShaper:
    """Estimate hourly demand for a synthetic delivery-market scenario."""

    def __init__(self, city_size: Tuple[float, float]):
        self.id = 1
        self.city_size = city_size
        self.customers: List[Customer] = []
        self.delivery_vehicles: List[Vehicle] = []
        self.incentive_mechanisms: List[IncentiveMechanism] = []
        self.demand_history: List[Dict[str, Any]] = []
        self.market_metrics = {
            "total_demand": 0.0,
            "peak_demand": 0.0,
            "off_peak_demand": 0.0,
            "demand_volatility": 0.0,
            "customer_satisfaction": 0.0,
            "operational_efficiency": 0.0,
            "cost_savings": 0.0,
        }

    def add_customer(self, customer: Customer) -> None:
        """Include a customer in the demand calculation."""
        if len(customer.base_demand_pattern) != 24:
            raise ValueError("Each customer must have exactly 24 hourly demand values.")
        self.customers.append(customer)

    def add_vehicle(self, vehicle: Vehicle) -> None:
        """Register a vehicle in the illustrative scenario."""
        self.delivery_vehicles.append(vehicle)

    def add_incentive_mechanism(self, mechanism: IncentiveMechanism) -> None:
        """Register an incentive description for the scenario."""
        self.incentive_mechanisms.append(mechanism)

    def analyze_demand_patterns(self) -> Dict[str, Any]:
        """Aggregate customer demand into a stochastic 24-hour profile."""
        hourly_demand = np.zeros(24)

        for customer in self.customers:
            for hour in range(24):
                # Random perturbations represent uncertainty in a synthetic profile.
                hour_demand = customer.base_demand_pattern[hour] + random.uniform(-0.2, 0.2)

                if hour in [8, 9, 17, 18]:
                    hour_demand *= 1.5
                elif hour in [13, 14]:
                    hour_demand *= 0.8

                hour_demand *= 1.0 + customer.price_sensitivity * 0.2
                hour_demand *= 1.0 + customer.time_preference * 0.1
                hour_demand *= 1.0 + customer.loyalty_score * 0.1
                hour_demand *= 1.0 + customer.responsiveness * 0.1

                # Bounds keep individual simulated values within the model's range.
                hour_demand = min(5.0, max(0.1, hour_demand))
                hourly_demand[hour] += hour_demand

        total_demand = float(np.sum(hourly_demand))
        peak_demand = float(np.max(hourly_demand)) if self.customers else 0.0
        off_peak_demand = float(np.min(hourly_demand)) if self.customers else 0.0
        mean_demand = float(np.mean(hourly_demand))
        demand_volatility = (
            float(np.std(hourly_demand) / mean_demand) if mean_demand > 0 else 0.0
        )

        analysis = {
            "hourly_demand": hourly_demand.tolist(),
            "total_demand": total_demand,
            "peak_demand": peak_demand,
            "off_peak_demand": off_peak_demand,
            "demand_volatility": demand_volatility,
        }
        self.demand_history.append(
            {
                "timestamp": datetime.now(),
                **analysis,
            }
        )
        return analysis


print("Market-shaping scenario components are ready.")

In [ ]:
print("=" * 60)
print("INITIALIZING THE PROACTIVE DEMAND-SHAPING SCENARIO")
print("=" * 60)

market_shaper = MarketShaper(city_size=(100, 100))

print("Market shaper status:")
print(f"  City size: {market_shaper.city_size}")
print(f"  Initial customers: {len(market_shaper.customers)}")
print(f"  Available vehicles: {len(market_shaper.delivery_vehicles)}")
print(f"  Incentive mechanisms: {len(market_shaper.incentive_mechanisms)}")

print("\nIllustrative holiday-season incentive mechanisms:")
market_shaper.add_incentive_mechanism(
    IncentiveMechanism(
        id=1,
        name="Peak Period Discount",
        type="pricing",
        target_segment=["all"],
        incentive_value=0.15,
        time_window=(16, 20),
        expected_impact=0.43,
        implementation_cost=500.0,
    )
)
market_shaper.add_incentive_mechanism(
    IncentiveMechanism(
        id=2,
        name="Off-Peak Consolidation",
        type="suggestion",
        target_segment=["all"],
        incentive_value=0.28,
        time_window=(13, 17),
        expected_impact=0.28,
        implementation_cost=300.0,
    )
)
market_shaper.add_incentive_mechanism(
    IncentiveMechanism(
        id=3,
        name="Neighborhood Consolidation",
        type="nudge",
        target_segment=["middle", "low_income"],
        incentive_value=0.73,
        time_window=(13, 17),
        expected_impact=0.73,
        implementation_cost=200.0,
    )
)

print(f"  Registered {len(market_shaper.incentive_mechanisms)} scenario mechanisms.")

In [ ]:
print("\nGenerating the synthetic holiday-season customer base:")

holiday_demand_multiplier = 3.4

for i in range(50):
    x = random.uniform(10, 90)
    y = random.uniform(10, 90)
    distance_from_center = np.sqrt((x - 50) ** 2 + (y - 50) ** 2)

    if distance_from_center < 20:
        base_demand = [
            random.uniform(2, 6) * holiday_demand_multiplier for _ in range(24)
        ]
        time_preference = random.uniform(0.0, 1.0)
        price_sensitivity = random.uniform(0.6, 1.0)
        loyalty_score = random.uniform(0.3, 0.9)
    elif distance_from_center < 40:
        base_demand = [
            random.uniform(1, 4) * holiday_demand_multiplier for _ in range(24)
        ]
        time_preference = random.uniform(0.0, 1.0)
        price_sensitivity = random.uniform(0.5, 0.9)
        loyalty_score = random.uniform(0.2, 0.8)
    else:
        base_demand = [
            random.uniform(0.5, 3) * holiday_demand_multiplier for _ in range(24)
        ]
        time_preference = random.uniform(0.0, 1.0)
        price_sensitivity = random.uniform(0.4, 0.8)
        loyalty_score = random.uniform(0.1, 0.7)

    customer = Customer(
        id=i + 1,
        location=(x, y),
        base_demand_pattern=base_demand,
        price_sensitivity=price_sensitivity,
        time_preference=time_preference,
        loyalty_score=loyalty_score,
        responsiveness=random.uniform(0.5, 1.0),
        demographic_profile={
            "age_group": np.random.choice(
                ["18-25", "26-35", "36-45", "46-55", "56-65", "66-75"]
            ),
            "income_bracket": np.random.choice(
                ["low", "middle", "high", "very_high"]
            ),
            "family_size": int(np.random.randint(1, 6)),
        },
    )
    market_shaper.add_customer(customer)

print(f"  Generated {len(market_shaper.customers)} customers.")

for i in range(10):
    vehicle = Vehicle(
        id=i + 1,
        capacity=random.uniform(50, 100),
        cost_per_km=random.uniform(0.5, 2.0),
        location=(random.uniform(10, 90), random.uniform(10, 90)),
    )
    market_shaper.add_vehicle(vehicle)

print(f"  Registered {len(market_shaper.delivery_vehicles)} illustrative vehicles.")

In [ ]:
print("\nBASELINE HOURLY DEMAND ANALYSIS")
baseline_analysis = market_shaper.analyze_demand_patterns()
hourly_demand = np.asarray(baseline_analysis["hourly_demand"], dtype=float)

print(f"  Total daily demand: {baseline_analysis['total_demand']:.1f} packages")
print(f"  Maximum hourly demand: {baseline_analysis['peak_demand']:.1f} packages")
print(f"  Minimum hourly demand: {baseline_analysis['off_peak_demand']:.1f} packages")
print(f"  Demand volatility (coefficient of variation): {baseline_analysis['demand_volatility']:.3f}")

peak_hour = int(np.argmax(hourly_demand))
off_peak_hour = int(np.argmin(hourly_demand))
print(f"  Busiest hour: {peak_hour:02d}:00")
print(f"  Quietest hour: {off_peak_hour:02d}:00")

if baseline_analysis["off_peak_demand"] > 0:
    peak_to_off_peak_ratio = (
        baseline_analysis["peak_demand"] / baseline_analysis["off_peak_demand"]
    )
    print(f"  Maximum-to-minimum hourly demand ratio: {peak_to_off_peak_ratio:.2f}x")
else:
    print("  Maximum-to-minimum hourly demand ratio: undefined (zero minimum demand)")

In [ ]:
print("\nSIMULATING THE DEMAND-SHAPING INTERVENTION")

# The bins 16 and 17 represent 4:00–5:00 PM and 5:00–6:00 PM.
peak_period_hours = np.array([16, 17])
off_peak_hours = np.array([hour for hour in range(24) if hour not in peak_period_hours])
modified_hourly_demand = hourly_demand.copy()

modified_hourly_demand[peak_period_hours] *= 1.0 - 0.43
modified_hourly_demand[off_peak_hours] *= 1.0 + 0.28

original_total = float(np.sum(hourly_demand))
modified_total = float(np.sum(modified_hourly_demand))
original_peak = float(np.max(hourly_demand))
modified_peak = float(np.max(modified_hourly_demand))

original_peak_period_total = float(np.sum(hourly_demand[peak_period_hours]))
modified_peak_period_total = float(np.sum(modified_hourly_demand[peak_period_hours]))
original_off_peak_total = float(np.sum(hourly_demand[off_peak_hours]))
modified_off_peak_total = float(np.sum(modified_hourly_demand[off_peak_hours]))

peak_reduction = (
    (original_peak_period_total - modified_peak_period_total)
    / original_peak_period_total
    * 100
    if original_peak_period_total > 0
    else 0.0
)
off_peak_increase = (
    (modified_off_peak_total - original_off_peak_total)
    / original_off_peak_total
    * 100
    if original_off_peak_total > 0
    else 0.0
)
maximum_hourly_peak_change = (
    (original_peak - modified_peak) / original_peak * 100
    if original_peak > 0
    else 0.0
)
total_demand_change = (
    (modified_total - original_total) / original_total * 100
    if original_total > 0
    else 0.0
)

print(f"  Demand in the 4:00–6:00 PM period: {original_peak_period_total:.1f} to {modified_peak_period_total:.1f} packages")
print(f"  Reduction in that period: {peak_reduction:.1f}%")
print(f"  Demand outside that period: {original_off_peak_total:.1f} to {modified_off_peak_total:.1f} packages")
print(f"  Increase outside that period: {off_peak_increase:.1f}%")
print(f"  Maximum hourly demand: {original_peak:.1f} to {modified_peak:.1f} packages")
print(f"  Change in maximum hourly demand: {maximum_hourly_peak_change:.1f}%")
print(f"  Total daily demand change: {total_demand_change:.1f}%")

vehicle_capacity = 80
vehicle_cost = 500
vehicles_needed_original = int(np.ceil(original_total / vehicle_capacity))
vehicles_needed_modified = int(np.ceil(modified_total / vehicle_capacity))
cost_savings = (vehicles_needed_original - vehicles_needed_modified) * vehicle_cost
emission_reduction = 0.5 * peak_reduction

print("\nILLUSTRATIVE OPERATING METRICS")
print(f"  Estimated vehicles before: {vehicles_needed_original}")
print(f"  Estimated vehicles after: {vehicles_needed_modified}")
print(f"  Estimated vehicle-cost savings: ${cost_savings:,.0f}")
print(f"  Emissions proxy: {emission_reduction:.1f} percentage points")
print("  Customer satisfaction: not modeled")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle(
    "Synthetic Demand-Shaping Scenario",
    fontsize=16,
    fontweight="bold",
)

hours = np.arange(24)
ax1 = axes[0, 0]
ax1.plot(hours, hourly_demand, color="tab:red", marker="o", label="Baseline")
ax1.plot(
    hours,
    modified_hourly_demand,
    color="tab:blue",
    marker="s",
    label="After assumed intervention",
)
ax1.axvspan(16, 18, alpha=0.15, color="tab:red", label="Target period: 4:00–6:00 PM")
ax1.set_xlabel("Hour of day")
ax1.set_ylabel("Demand (packages per hour)")
ax1.set_title("Hourly demand profile")
ax1.set_xticks([0, 6, 12, 18, 23])
ax1.set_xticklabels(["00:00", "06:00", "12:00", "18:00", "23:00"])
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2 = axes[0, 1]
period_labels = ["4:00–6:00 PM", "All other hours"]
original_period_totals = [original_peak_period_total, original_off_peak_total]
modified_period_totals = [modified_peak_period_total, modified_off_peak_total]
positions = np.arange(len(period_labels))
bar_width = 0.35
ax2.bar(
    positions - bar_width / 2,
    original_period_totals,
    bar_width,
    label="Baseline",
    color="tab:red",
    alpha=0.75,
)
ax2.bar(
    positions + bar_width / 2,
    modified_period_totals,
    bar_width,
    label="After assumed intervention",
    color="tab:blue",
    alpha=0.75,
)
ax2.set_xlabel("Hourly-bin group")
ax2.set_ylabel("Demand (packages)")
ax2.set_title("Demand totals by period")
ax2.set_xticks(positions)
ax2.set_xticklabels(period_labels)
ax2.grid(axis="y", alpha=0.3)
ax2.legend()

ax3 = axes[1, 0]
change_labels = ["Target-period reduction", "Other-hours increase", "Total-demand change"]
change_values = [peak_reduction, off_peak_increase, total_demand_change]
change_colors = ["tab:green", "tab:orange", "tab:purple"]
ax3.bar(change_labels, change_values, color=change_colors, alpha=0.75)
ax3.axhline(0, color="black", linewidth=0.8)
ax3.set_ylabel("Change (%)")
ax3.set_title("Demand changes under the stated assumptions")
ax3.tick_params(axis="x", rotation=15)
ax3.grid(axis="y", alpha=0.3)

ax4 = axes[1, 1]
vehicle_labels = ["Baseline", "After intervention"]
vehicle_values = [vehicles_needed_original, vehicles_needed_modified]
ax4.bar(vehicle_labels, vehicle_values, color=["tab:red", "tab:blue"], alpha=0.75)
ax4.set_ylabel("Estimated vehicles (80 packages each)")
ax4.set_title(f"Vehicle estimate; assumed cost = ${vehicle_cost:,} per vehicle")
ax4.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("PROACTIVE DEMAND-SHAPING SCENARIO: SUMMARY")
print("=" * 60)

print("\nSIMULATED COMPARISON")
print(f"  Reduction in the 4:00–6:00 PM period: {peak_reduction:.1f}%")
print(f"  Increase outside that period: {off_peak_increase:.1f}%")
print(f"  Change in maximum hourly demand: {maximum_hourly_peak_change:.1f}%")
print(f"  Change in total daily demand: {total_demand_change:.1f}%")
print(f"  Estimated vehicle-cost savings: ${cost_savings:,.0f}")
print(f"  Emissions proxy: {emission_reduction:.1f} percentage points")

print("\nREFERENCE CLAIMS NOT ESTIMATED BY THIS SIMULATION")
print("  Peak-demand reduction, off-peak adoption, and emissions outcomes depend on scenario assumptions.")
print("  The $18,200 savings claim is not derived from this model's vehicle-cost calculation.")
print("  Customer satisfaction and incentive adoption are not modeled.")

if peak_reduction >= 0 and off_peak_increase >= 0:
    interpretation = (
        "The assumed intervention reduces demand in its target period and increases "
        "demand in the other hourly bins."
    )
else:
    interpretation = (
        "At least one period-level comparison does not follow the expected direction; "
        "inspect the inputs and intervention calculations."
    )
print(f"\nInterpretation: {interpretation}")
print("These are scenario calculations, not measured causal effects.")

## Strategic role, benefits, and limitations

### Why proactive demand shaping matters

Demand shaping extends operational planning by considering how delivery choices and incentives may influence when customers request service. The simulation in this notebook represents that idea with specified changes to hourly demand. It does not forecast behavior or estimate the causal effect of an incentive; those tasks would require suitable customer and operational data.

### Potential benefits

- Smoother demand can make capacity planning easier when customers genuinely shift orders across time periods.
- Incentives and delivery suggestions can align customer choices with available service capacity when they are transparent and worthwhile to customers.
- Coordinating demand and operations can support more efficient use of vehicles and fulfillment capacity.

### Limitations and risks

- Realistic implementation requires reliable demand data and evidence about customer responses.
- Incentives may redistribute demand, create new demand, or have little effect; the simple multipliers here cannot distinguish these outcomes.
- Customer privacy, fairness, accessibility, transparency, and applicable pricing regulations must be considered in any real deployment.
- Vehicle counts based on aggregate daily demand do not guarantee that vehicles can meet hourly, geographic, or service-time requirements.
- The emissions proxy is not a substitute for route-level distance, vehicle, energy, or emissions data.

### When this approach may be useful

A demand-shaping analysis may be useful when a delivery network has recurring capacity imbalances, reliable customer-demand data, and ethically designed options that customers can freely accept or decline. A practical evaluation should distinguish assumptions from measured outcomes and validate service quality, customer response, cost, and environmental effects against observed data.

### Key concepts

- **Choice architecture:** How available options are presented to support informed customer decisions.
- **Dynamic incentives:** Prices or benefits that vary with conditions such as delivery time or capacity.
- **Predictive demand management:** Using forecasts to anticipate likely demand and plan capacity.
- **Market coordination:** Aligning customer options, fulfillment capacity, and delivery operations.
- **Ethical demand shaping:** Designing interventions that are transparent, fair, and beneficial to customers as well as the operator.